# Milestone 5 — Paged KV cache

Fixed-size KV **blocks** + a **block table**. Same memory budget: **reserved**
(worst-case at admit) vs **paged** (grow as tokens are produced).

In [ ]:
!git clone https://github.com/Jayaprakash-030/tiny-inference-engine.git
%cd tiny-inference-engine
!pip install -q -e .

In [ ]:
from engine import load
from engine.paged import (
    check_paged_matches_cached,
    compare_block_budget,
    sweep_block_budget,
)
from engine.results import save

rt = load()
rt.describe()

### Correctness

In [ ]:
assert check_paged_matches_cached(rt)

### Reserved vs paged under one budget, then sweep + save

In [ ]:
reserved, paged = compare_block_budget(
    rt, n_blocks=32, n_requests=16, max_new_tokens=64,
)
save([reserved, paged])

In [ ]:
m5 = sweep_block_budget(
    rt,
    block_budgets=(16, 32, 48, 64),
    n_requests=16,
    max_new_tokens=64,
)
save(m5)

### Plot completed / rejected / peak concurrent

In [ ]:
import matplotlib.pyplot as plt
from pathlib import Path

Path("benchmarks/plots").mkdir(parents=True, exist_ok=True)

reserved_rows = [r for r in m5 if r["mode"] == "reserved"]
paged_rows = [r for r in m5 if r["mode"] == "paged"]
budgets = [r["n_blocks"] for r in reserved_rows]

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, key, title in [
    (axes[0], "completed", "Completed requests"),
    (axes[1], "rejected", "Rejected requests"),
    (axes[2], "peak_concurrent", "Peak concurrent"),
]:
    ax.plot(budgets, [r[key] for r in reserved_rows], marker="o", label="reserved")
    ax.plot(budgets, [r[key] for r in paged_rows], marker="o", label="paged")
    ax.set_title(title)
    ax.set_xlabel("n_blocks (budget)")
    ax.legend()
    ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig("benchmarks/plots/m5_paged_kv.png", dpi=140)
plt.show()